# 🔍 Floor Plan Analysis: Multi-Format Inference & Evaluation
### Parse JPG, PNG, and SVG Blueprints with Fine-Tuned Qwen3-VL 8B

This notebook demonstrates:
1. Loading the fine-tuned LoRA adapter from Google Drive.
2. Analyzing floor plans in **JPG, PNG, and SVG vector formats**.
3. Extracting structured metadata: **rooms, length, width, doors, windows, and normalized coordinates**.
4. Visualizing detected bounding boxes and dimensions overlaid on the original blueprints.
5. Evaluating Mean Intersection-over-Union (mIoU) and detection accuracy on your validation dataset.

In [ ]:
# Step 1: Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/floorplan_reader_project'
except Exception:
    PROJECT_DIR = './data/colab_run'

LORA_MODEL_DIR = f'{PROJECT_DIR}/models/qwen3_vl_floorplan_lora'
print(f"Loading LoRA weights from: {LORA_MODEL_DIR}")

In [ ]:
# Step 2: Install dependencies
!pip install -q unsloth trl peft bitsandbytes cairosvg svglib reportlab pydantic matplotlib

In [ ]:
# Step 3: Load Fine-Tuned Model and Define Inference Pipeline
import os, json, re, io
import torch
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from unsloth import FastVisionModel

# Color palettes for room visualization
ROOM_COLORS = {
    'bedroom': '#3498db', 'master_bedroom': '#2980b9',
    'living_room': '#2ecc71', 'kitchen': '#f1c40f',
    'bathroom': '#9b59b6', 'hallway': '#95a5a6',
    'balcony': '#1abc9c', 'dining_room': '#e67e22',
    'office': '#34495e', 'storage': '#7f8c8d',
    'room': '#16a085'
}

print(f'Checking for LoRA weights in: {LORA_MODEL_DIR}...')
if os.path.exists(LORA_MODEL_DIR):
    print('Loading fine-tuned LoRA model from Google Drive...')
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name=LORA_MODEL_DIR,
        load_in_4bit=True,
    )
    FastVisionModel.for_inference(model)
    print('Model ready on GPU!')
else:
    print('Note: LORA_MODEL_DIR not found yet. Loading base model for demonstration...')
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name='unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit',
        load_in_4bit=True,
    )
    FastVisionModel.for_inference(model)

def load_floorplan_image(img_path):
    '''Load JPG, PNG, or convert SVG to high-res PIL Image.'''
    p_str = str(img_path).lower()
    if p_str.endswith('.svg'):
        try:
            import cairosvg
            png_bytes = cairosvg.svg2png(url=str(img_path), background_color='white')
            img = Image.open(io.BytesIO(png_bytes))
        except Exception:
            from svglib.svglib import svg2rlg
            from reportlab.graphics import renderPM
            drawing = svg2rlg(str(img_path))
            buf = io.BytesIO()
            renderPM.drawToFile(drawing, buf, fmt='PNG')
            buf.seek(0)
            img = Image.open(buf)
    else:
        img = Image.open(img_path)
    return img.convert('RGB')

def predict_floorplan(image, pixels_per_meter=None):
    '''Run VLM inference on floor plan and return structured analysis.'''
    w, h = image.size
    prompt = (
        'Analyze this architectural floor plan drawing. Detect all rooms, doors, and windows.\n'
        'Output a valid JSON object with the following schema:\n'
        '{\n'
        '  "rooms": [{"id": "room_1", "name": "bedroom", "box_2d": [ymin, xmin, ymax, xmax], "detected_label_text": "optional text"}],\n'
        '  "doors": [{"id": "door_1", "type": "single_swing", "box_2d": [ymin, xmin, ymax, xmax]}],\n'
        '  "windows": [{"id": "win_1", "type": "standard", "box_2d": [ymin, xmin, ymax, xmax]}]\n'
        '}\n'
        'Use normalized coordinates [ymin, xmin, ymax, xmax] scaled 0 to 1000.'
    )
    messages = [
        {'role': 'user', 'content': [{'type': 'image', 'image': image}, {'type': 'text', 'text': prompt}]}
    ]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(images=[image], text=[input_text], return_tensors='pt').to('cuda')
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=1024, temperature=0.1)
    
    gen_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    # Extract JSON
    match = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', gen_text) or re.search(r'\{[\s\S]*\}', gen_text)
    clean_json = match.group(1) if match and '```' in gen_text else (match.group(0) if match else '{}')
    try:
        data = json.loads(clean_json)
    except Exception:
        # Clean trailing commas
        repaired = re.sub(r',\s*([\]\}])', r'\1', clean_json)
        data = json.loads(repaired)
        
    # Calculate dimensions and metadata
    for r in data.get('rooms', []):
        box = r.get('box_2d', [0, 0, 0, 0])
        ymin, xmin, ymax, xmax = box
        norm_w = max(0, xmax - xmin)
        norm_h = max(0, ymax - ymin)
        r['norm_length'] = max(norm_w, norm_h)
        r['norm_width'] = min(norm_w, norm_h)
        r['area_percentage'] = round((norm_w * norm_h / 1_000_000) * 100, 2)
        if pixels_per_meter:
            px_w = (norm_w / 1000) * w
            px_h = (norm_h / 1000) * h
            r['real_length_m'] = round(max(px_w, px_h) / pixels_per_meter, 2)
            r['real_width_m'] = round(min(px_w, px_h) / pixels_per_meter, 2)
            
    return data

print('Inference pipeline ready!')


### Step 4: Run Inference on Any Floor Plan (JPG, PNG, or SVG)

In [ ]:
# Step 4: Run Inference and Plot Visual Overlay
import glob

# Find a test image from your processed CubiCasa images or specify any image path
test_images = glob.glob(f'{PROJECT_DIR}/images/*/*.png')
if test_images:
    sample_path = test_images[0]
    print(f'Testing on sample: {sample_path}')
else:
    # Fallback to create a test floor plan image if none found
    sample_path = '/content/sample_test_plan.png'
    test_img = Image.new('RGB', (800, 600), color=(250, 250, 250))
    test_img.save(sample_path)

raw_image = load_floorplan_image(sample_path)
w, h = raw_image.size

# Run Model Prediction
analysis = predict_floorplan(raw_image, pixels_per_meter=50.0)

# Plot side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(raw_image)
axes[0].set_title('Original Floor Plan', fontsize=14)
axes[0].axis('off')

axes[1].imshow(raw_image)
# Overlay Rooms
for r in analysis.get('rooms', []):
    box = r.get('box_2d', [])
    if len(box) == 4:
        ymin, xmin, ymax, xmax = [v / 1000.0 for v in box]
        color = ROOM_COLORS.get(r.get('name', '').lower(), '#34495e')
        rect = patches.Rectangle(
            (xmin * w, ymin * h), (xmax - xmin) * w, (ymax - ymin) * h,
            linewidth=2.5, edgecolor=color, facecolor=color, alpha=0.25
        )
        axes[1].add_patch(rect)
        dim_txt = f"{r['norm_length']:.0f}x{r['norm_width']:.0f}"
        if 'real_length_m' in r:
            dim_txt = f"{r['real_length_m']}x{r['real_width_m']}m"
        label = f"{r.get('name', 'room').title()}\n{dim_txt} ({r['area_percentage']:.1f}%)"
        axes[1].text(xmin * w + 5, ymin * h + 15, label, fontsize=9, color='white',
                    weight='bold', bbox=dict(facecolor='black', alpha=0.7, pad=2, edgecolor='none'))

# Overlay Doors (Red)
for d in analysis.get('doors', []):
    box = d.get('box_2d', [])
    if len(box) == 4:
        ymin, xmin, ymax, xmax = [v / 1000.0 for v in box]
        rect = patches.Rectangle((xmin * w, ymin * h), max(8, (xmax - xmin) * w), max(8, (ymax - ymin) * h),
                                 linewidth=2, edgecolor='red', facecolor='red', alpha=0.8)
        axes[1].add_patch(rect)

# Overlay Windows (Cyan)
for win in analysis.get('windows', []):
    box = win.get('box_2d', [])
    if len(box) == 4:
        ymin, xmin, ymax, xmax = [v / 1000.0 for v in box]
        rect = patches.Rectangle((xmin * w, ymin * h), max(8, (xmax - xmin) * w), max(8, (ymax - ymin) * h),
                                 linewidth=2, edgecolor='cyan', facecolor='cyan', alpha=0.8)
        axes[1].add_patch(rect)

axes[1].set_title(f"Detected Layout: {len(analysis.get('rooms', []))} Rooms, {len(analysis.get('doors', []))} Doors, {len(analysis.get('windows', []))} Windows", fontsize=14)
axes[1].axis('off')
plt.tight_layout()
plt.show()

# Print Structured JSON Output
print('\nStructured JSON Output:')
print(json.dumps(analysis, indent=2))


### Step 5: Inspect Structured JSON Output

In [ ]:
import json
print(json.dumps(analysis.to_clean_dict(), indent=2))

### Step 6: Quantitative Validation Set Evaluation (mIoU and Precision/Recall)

In [ ]:
# Step 5: Quantitative Validation Evaluation (mIoU and Precision/Recall)
def calculate_iou(boxA, boxB):
    '''Compute Intersection-over-Union between two [ymin, xmin, ymax, xmax] boxes.'''
    yA = max(boxA[0], boxB[0])
    xA = max(boxA[1], boxB[1])
    yB = min(boxA[2], boxB[2])
    xB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = max(0, boxA[3] - boxA[1]) * max(0, boxA[2] - boxA[0])
    boxBArea = max(0, boxB[3] - boxB[1]) * max(0, boxB[2] - boxB[0])
    denom = float(boxAArea + boxBArea - interArea)
    return interArea / denom if denom > 0 else 0.0

def evaluate_floorplan_sample(gt_dict, pred_dict, iou_threshold=0.5):
    gt_rooms = gt_dict.get('rooms', [])
    pred_rooms = pred_dict.get('rooms', [])
    matched_gt = set()
    ious = []
    
    for pr in pred_rooms:
        best_iou = 0.0
        best_idx = -1
        for idx, gt in enumerate(gt_rooms):
            if idx in matched_gt:
                continue
            score = calculate_iou(pr.get('box_2d', [0,0,0,0]), gt.get('box_2d', [0,0,0,0]))
            if score > best_iou:
                best_iou = score
                best_idx = idx
        if best_iou >= iou_threshold:
            matched_gt.add(best_idx)
            ious.append(best_iou)
            
    precision = len(matched_gt) / max(1, len(pred_rooms))
    recall = len(matched_gt) / max(1, len(gt_rooms))
    mean_iou = sum(ious) / max(1, len(ious))
    return {'precision': precision, 'recall': recall, 'mean_iou': mean_iou}

print('Evaluation helper ready! Call evaluate_floorplan_sample(ground_truth_json, predicted_json)')
